# Set up Open Managed Agents on Cloudflare

Use this notebook from a browser-based JavaScript kernel to provision the Cloudflare resources used by Open Managed Agents (OMA). The resource names and deployment handoff follow OMA's `scripts/setup-cf.sh` and self-host deployment guide.

**Prerequisite:** a Cloudflare account on the Workers Paid plan. OMA uses Durable Objects, R2, and Containers, which are not available on the Free plan.

> This notebook creates D1 databases, a KV namespace, and R2 buckets. It does **not** build or upload OMA's three Worker bundles, apply repository migrations, set Worker secrets, or deploy the Workers. Those steps require the OMA source/build environment and are described at the end. Cloudflare API access must already be available to this kernel; a normal browser may block direct API requests due to CORS.

## Prepare access

Create a short-lived API token scoped to this account with **D1:Edit**, **Workers KV Storage:Edit**, and **R2:Edit**. This notebook never prints or stores the token in cell source or outputs, but it remains in the running kernel's memory. Do not share the notebook while the kernel is active; reset the kernel when finished. Do not use an account-wide Global API Key.

The cells list existing resources before creating anything, so re-running after a partial failure reuses what was already created. Resource creation changes your Cloudflare account and may incur charges.

In [ ]:
globalThis.omaSetup ??= {};
globalThis.omaSetup.accountId = globalThis.prompt?.('Cloudflare account ID (32-character hex)')?.trim();
if (!/^[a-f0-9]{32}$/i.test(globalThis.omaSetup.accountId ?? '')) {
  throw new Error('Enter a valid 32-character Cloudflare account ID.');
}

globalThis.omaSetup.token = globalThis.prompt?.('Short-lived Cloudflare API token (input is not masked)')?.trim();
if (!globalThis.omaSetup.token) {
  throw new Error('A Cloudflare API token is required. Ensure this browser kernel supports prompt().');
}

globalThis.omaSetup.request = async (path, options = {}) => {
  const response = await fetch(`https://api.cloudflare.com/client/v4${path}`, {
    ...options,
    headers: {
      Authorization: `Bearer ${globalThis.omaSetup.token}`,
      'Content-Type': 'application/json',
      ...options.headers
    },
    body: options.body === undefined ? undefined : JSON.stringify(options.body)
  });
  const payload = await response.json();
  if (!response.ok || payload.success === false) {
    const details = (payload.errors ?? []).map(error => `${error.code ?? ''} ${error.message ?? ''}`.trim()).join('; ');
    throw new Error(`Cloudflare API ${options.method ?? 'GET'} ${path} failed (${response.status}): ${details || response.statusText}`);
  }
  return payload.result;
};

'Cloudflare API client ready; token omitted.'

## Provision OMA's storage resources

This matches the single-account self-host baseline: two D1 databases, one KV namespace, and four R2 buckets. Existing resources with these exact names are reused; other resources are left untouched.

In [ ]:
await (async () => {
  const account = `/accounts/${globalThis.omaSetup.accountId}`;
  const [databases, namespaces, r2Result] = await Promise.all([
    globalThis.omaSetup.request(`${account}/d1/database`),
    globalThis.omaSetup.request(`${account}/storage/kv/namespaces?per_page=1000`),
    globalThis.omaSetup.request(`${account}/r2/buckets`)
  ]);
  const buckets = Array.isArray(r2Result) ? r2Result : r2Result.buckets;
  if (!Array.isArray(databases) || !Array.isArray(namespaces) || !Array.isArray(buckets)) {
    throw new Error('Unexpected Cloudflare list response; no resources were created.');
  }

  const ensureD1 = async name => {
    const existing = databases.find(database => database.name === name);
    if (existing) return { name, id: existing.uuid, created: false };
    const created = await globalThis.omaSetup.request(`${account}/d1/database`, {
      method: 'POST', body: { name }
    });
    return { name, id: created.uuid, created: true };
  };
  const ensureKV = async title => {
    const existing = namespaces.find(namespace => namespace.title === title);
    if (existing) return { title, id: existing.id, created: false };
    const created = await globalThis.omaSetup.request(`${account}/storage/kv/namespaces`, {
      method: 'POST', body: { title }
    });
    return { title, id: created.id, created: true };
  };
  const ensureR2 = async name => {
    const existing = buckets.find(bucket => bucket.name === name);
    if (existing) return { name, created: false };
    await globalThis.omaSetup.request(`${account}/r2/buckets`, {
      method: 'POST', body: { name }
    });
    return { name, created: true };
  };

  const d1 = {};
  for (const name of ['openma-auth', 'openma-integrations']) d1[name] = await ensureD1(name);
  const kv = await ensureKV('CONFIG_KV');
  const r2 = {};
  for (const name of [
    'managed-agents-files',
    'managed-agents-workspace',
    'managed-agents-memory',
    'managed-agents-backups'
  ]) r2[name] = await ensureR2(name);

  for (const resource of [...Object.values(d1), kv, ...Object.values(r2)]) {
    if (resource.created) console.log(`Created ${resource.name ?? resource.title}`);
    else console.log(`Reusing ${resource.name ?? resource.title}`);
  }
  if (Object.values(d1).some(database => !database.id) || !kv.id) {
    throw new Error('Cloudflare returned a resource without an ID; inspect the account before continuing.');
  }

  globalThis.omaSetup.manifest = {
    accountId: globalThis.omaSetup.accountId,
    d1,
    kv,
    r2
  };
  globalThis.omaSetup.manifest
})();

## Verify and prepare the config handoff

Re-list the expected names and inspect the manifest. It contains resource IDs and names only—never API credentials. Use these values when patching OMA's Wrangler files; the notebook cannot edit the source checkout from the browser.

In [ ]:
await (async () => {
  if (!globalThis.omaSetup.manifest) throw new Error('Run the provisioning cell first.');
  const account = `/accounts/${globalThis.omaSetup.accountId}`;
  const [databases, namespaces, r2Result] = await Promise.all([
    globalThis.omaSetup.request(`${account}/d1/database`),
    globalThis.omaSetup.request(`${account}/storage/kv/namespaces?per_page=1000`),
    globalThis.omaSetup.request(`${account}/r2/buckets`)
  ]);
  const buckets = Array.isArray(r2Result) ? r2Result : r2Result.buckets;
  const expectedD1 = ['openma-auth', 'openma-integrations'];
  const expectedR2 = [
    'managed-agents-files', 'managed-agents-workspace',
    'managed-agents-memory', 'managed-agents-backups'
  ];
  const missing = [
    ...expectedD1.filter(name => !databases.some(database => database.name === name)),
    ...(!namespaces.some(namespace => namespace.title === 'CONFIG_KV') ? ['KV CONFIG_KV'] : []),
    ...expectedR2.filter(name => !buckets.some(bucket => bucket.name === name))
  ];
  if (missing.length) throw new Error(`Verification failed; missing: ${missing.join(', ')}`);

  const manifest = globalThis.omaSetup.manifest;
  const integrationsOrigin = `https://managed-agents-integrations.${manifest.accountId.slice(0, 8)}.workers.dev`;
  globalThis.omaSetup.wranglerHandoff = {
    main: {
      d1_databases: [
        { binding: 'MAIN_DB', database_name: 'openma-auth', database_id: manifest.d1['openma-auth'].id },
        { binding: 'INTEGRATIONS_DB', database_name: 'openma-integrations', database_id: manifest.d1['openma-integrations'].id }
      ],
      kv_namespaces: [{ binding: 'CONFIG_KV', id: manifest.kv.id }],
      vars: { INTEGRATIONS_ORIGIN: integrationsOrigin }
    },
    agent: {
      d1_databases: [{ binding: 'MAIN_DB', database_name: 'openma-auth', database_id: manifest.d1['openma-auth'].id }],
      kv_namespaces: [{ binding: 'CONFIG_KV', id: manifest.kv.id }],
      vars: {
        CLOUDFLARE_ACCOUNT_ID: manifest.accountId,
        MEMORY_BUCKET_NAME: 'managed-agents-memory',
        WORKSPACE_BUCKET_NAME: 'managed-agents-workspace',
        BACKUP_BUCKET_NAME: 'managed-agents-backups'
      }
    },
    integrations: {
      d1_databases: [
        { binding: 'MAIN_DB', database_name: 'openma-auth', database_id: manifest.d1['openma-auth'].id },
        { binding: 'INTEGRATIONS_DB', database_name: 'openma-integrations', database_id: manifest.d1['openma-integrations'].id }
      ],
      vars: { GATEWAY_ORIGIN: integrationsOrigin }
    },
    r2_bucket_names: Object.keys(manifest.r2)
  };

  console.log('Verification passed: all 7 expected resources exist.');
  console.log('Resource manifest (IDs are not credentials):');
  console.log(JSON.stringify(manifest, null, 2));
  console.log('Wrangler config handoff:');
  console.log(JSON.stringify(globalThis.omaSetup.wranglerHandoff, null, 2));
})();

## Finish deployment from the OMA source checkout

1. Patch the resource IDs and vars in `apps/main/wrangler.jsonc`, `apps/agent/wrangler.jsonc`, and `apps/integrations/wrangler.jsonc` using the handoff above. Also keep the configured R2 bucket bindings aligned with the names provisioned here.
2. From the OMA repository, follow `apps/docs/src/content/docs/self-host/deploy.mdx` and run `scripts/setup-cf.sh` in an environment with the source, Node/pnpm, and Wrangler. That step applies D1 migrations, generates the required Worker secrets, asks for the LLM provider key, deploys the workers, and creates the R2 notification after the queue consumer exists. Do not paste an LLM key into this notebook.
3. Review the generated bindings before deploying: the current setup script calls its main and agent D1 binding `AUTH_DB`, while the current top-level Wrangler configs use `MAIN_DB`. If unchanged, the script's binding patch can skip those entries and leave the placeholder ID in place. Ensure the `MAIN_DB` entries receive the `openma-auth` ID before running migrations/deploy.
4. The expected worker deployment order is integrations, agent, then main. After deployment, connect `managed-agents-memory` to the `managed-agents-memory-events` queue for `object-create` and `object-delete` notifications if setup did not do so successfully.
5. Open the main Worker URL and sign up; the first user becomes the owner of the first tenant.

After you have recorded the non-secret manifest, **reset the JavaScript kernel** to discard the API token from memory.

## Consume Claude Managed Agents APIs: mock routing and live OMA

The first example registers fake Managed Agents-shaped responses on cell-defined routes and exercises them without credentials or network calls. The second sends a real request to a deployed OMA API. For the live example, you need an existing Agent, a self-hosted Environment, and a workspace API key created from OMA Console → API Keys. Do not use your Cloudflare token or OMA's operator `API_KEY` secret here. Creating a Session and running a turn can consume model/runtime resources and incur charges.

### 1. Fake responses for routing tests

Run this cell to claim a notebook-specific prefix and register mock Agent, Environment, and Session handlers. Re-running it replaces these handlers; the fake IDs and responses are intentionally stable.

In [ ]:
globalThis.omaRouteMock ??= {};
globalThis.omaRouteMock.prefix = '/api/openma-claude-managed-agents/';
claimPrefix(globalThis.omaRouteMock.prefix);

app.get(`${globalThis.omaRouteMock.prefix}v1/agents/agent_mock_001`, c => c.json({
  id: 'agent_mock_001',
  type: 'agent',
  name: 'Notebook routing mock',
  model: { id: 'mock-model' },
  version: 1
}));

app.get(`${globalThis.omaRouteMock.prefix}v1/environments/env_mock_001`, c => c.json({
  id: 'env_mock_001',
  type: 'environment',
  name: 'Notebook routing mock environment',
  config: { type: 'self_hosted' }
}));

app.post(`${globalThis.omaRouteMock.prefix}v1/sessions`, async c => {
  const body = await c.req.json();
  if (body.agent !== 'agent_mock_001' || body.environment_id !== 'env_mock_001') {
    return c.json({ error: 'Unknown mock Agent or Environment ID' }, 400);
  }
  return c.json({
    id: 'sess_mock_001',
    type: 'session',
    agent: body.agent,
    environment_id: body.environment_id,
    title: body.title ?? 'Mock session',
    status: 'idle'
  }, 201);
});

'Mock Agent, Environment, and Session routes registered.'

Call the mock endpoints through `fetch()` just as a browser client would. These requests stay on the notebook route and never contact OMA or Anthropic.

In [ ]:
await (async () => {
  const base = `${globalThis.omaRouteMock.prefix}v1`;
  const getJson = async path => {
    const response = await fetch(`${base}${path}`);
    const body = await response.json();
    if (!response.ok) throw new Error(`Mock GET ${path} failed (${response.status}): ${JSON.stringify(body)}`);
    return body;
  };
  const agent = await getJson('/agents/agent_mock_001');
  const environment = await getJson('/environments/env_mock_001');
  const sessionResponse = await fetch(`${base}/sessions`, {
    method: 'POST',
    headers: { 'content-type': 'application/json' },
    body: JSON.stringify({
      agent: agent.id,
      environment_id: environment.id,
      title: 'Notebook mock routing test'
    })
  });
  const session = await sessionResponse.json();
  if (!sessionResponse.ok) throw new Error(`Mock POST /sessions failed (${sessionResponse.status}): ${JSON.stringify(session)}`);
  if (agent.type !== 'agent' || environment.type !== 'environment' || session.type !== 'session') {
    throw new Error('Mock route response did not match the expected resource types.');
  }
  console.log({ agent, environment, session });
  console.log('Claimed prefixes:', await claimedPrefixes());
})();

### 2. Real usage against your deployed OMA API

This uses OMA's Anthropic-compatible routes: `POST /v1/sessions`, then `POST /v1/sessions/{id}/messages` to create a real session and send its first turn. Enter your deployment's exact HTTPS base URL, existing Agent and self-hosted Environment IDs, and an `oma_...` workspace key from the Console API Keys page. The key is requested at runtime and is not written to the notebook; browser prompts may show the text as you type, so do not run this on a shared screen. Confirm the session creation explicitly; the turn response is an SSE stream. After the response body is collected, the notebook replays its data messages with one-second gaps for readability; this is paced playback, not incremental network delivery.

For endpoint details, see [OMA API guide](https://docs.openma.ai/build/api/). Browser cross-origin requests must be allowed by the deployment; if your browser blocks them, use a trusted same-origin backend proxy rather than weakening CORS or embedding credentials in a public page.

In [ ]:
await (async () => {
  const baseUrl = globalThis.prompt?.('Deployed OMA HTTPS base URL (for example, https://your-worker.example.workers.dev)')?.trim().replace(/\/+$/, '');
  const apiKey = globalThis.prompt?.('OMA workspace API key (oma_...)')?.trim();
  const agentId = globalThis.prompt?.('Existing Agent ID')?.trim();
  const environmentId = globalThis.prompt?.('Existing self-hosted Environment ID')?.trim();
  const firstTurn = globalThis.prompt?.('First user message for the new session')?.trim();
  if (!baseUrl || new URL(baseUrl).protocol !== 'https:' || !apiKey || !agentId || !environmentId || !firstTurn) {
    throw new Error('Enter an HTTPS OMA URL, API key, Agent ID, Environment ID, and non-empty user message.');
  }
  if (globalThis.prompt?.('This will create a real Session and run a model turn. Type RUN to continue.') !== 'RUN') {
    throw new Error('Live session creation cancelled.');
  }

  const headers = { 'content-type': 'application/json', 'x-api-key': apiKey };
  const createResponse = await fetch(`${baseUrl}/v1/sessions`, {
    method: 'POST',
    headers,
    body: JSON.stringify({ agent: agentId, environment_id: environmentId, title: 'Browser notebook session' })
  });
  if (!createResponse.ok) {
    throw new Error(`OMA session creation failed (${createResponse.status}): ${await createResponse.text()}`);
  }
  const session = await createResponse.json();
  if (typeof session.id !== 'string' || !session.id) throw new Error('OMA returned a session without an ID.');

  const turnResponse = await fetch(`${baseUrl}/v1/sessions/${encodeURIComponent(session.id)}/messages`, {
    method: 'POST',
    headers,
    body: JSON.stringify({ content: firstTurn })
  });
  if (!turnResponse.ok) {
    throw new Error(`OMA turn request failed (${turnResponse.status}) for session ${session.id}: ${await turnResponse.text()}`);
  }
  const sseText = await turnResponse.text();
  if (!sseText.trim()) throw new Error(`OMA returned an empty turn stream for session ${session.id}.`);
  globalThis.omaLiveResult = { sessionId: session.id, sseText };
  const sseMessages = sseText
    .split(/\r?\n\r?\n/)
    .map(frame =>
      frame
        .split(/\r?\n/)
        .filter(line => line.startsWith('data:'))
        .map(line => line.slice(5).replace(/^ /, ''))
        .join('\n')
    )
    .filter(data => data && data !== '[DONE]');
  console.log(`Replaying ${sseMessages.length} SSE messages with one-second gaps.`);
  for (let index = 0; index < sseMessages.length; index++) {
    if (index > 0) await new Promise(resolve => setTimeout(resolve, 1000));
    console.log(`SSE message ${index + 1}:`, sseMessages[index].slice(0, 500));
  }
  console.log({
    sessionId: session.id,
    streamContentType: turnResponse.headers.get('content-type'),
    streamPreview: sseText.slice(0, 4000)
  });
})();